# Découverte du robot

Ce carnet lit les trois capteurs du robot : la batterie, le lidar et la caméra. **Il ne fait pas bouger les roues.**

Exécutez les cellules une par une avec **Maj + Entrée**. Le code s'exécute sur la Raspberry Pi du robot ; les résultats s'affichent sous chaque cellule.

## Batterie

Les moteurs Dynamixel sont branchés directement sur la batterie (3 éléments Li-ion en série) et mesurent sa tension. Un élément est plein vers 4,2 V ; en dessous de 3,3 V, il faut arrêter et recharger.

In [ ]:
import statistics

import pypot.dynamixel as dxl

io = dxl.DxlIO("/dev/serial0", baudrate=57600)
try:
    voltage = statistics.median(io.get_present_voltage([1, 2, 4, 8]))
finally:
    io.close()  # libère le bus des moteurs pour les autres programmes
print(f"Batterie : {voltage:.1f} V, soit {voltage / 3:.2f} V par élément")

## Lidar

Le lidar tourne sur lui-même environ 7 fois par seconde et mesure la distance des obstacles tout autour du robot. `get_scan()` renvoie un tour complet : des angles en degrés (sens des aiguilles d'une montre) et des distances en mètres. Documentation : `docs/lidar.md` dans le dépôt.

In [ ]:
import math

import matplotlib.pyplot as plt
from holorobot.lidar import Lidar

with Lidar() as lidar:  # « with » arrête le moteur du lidar à la fin, même en cas d'erreur
    scan = lidar.get_scan()

angle, distance = scan.nearest()
print(f"{len(scan)} points ; obstacle le plus proche : {distance:.2f} m à {angle:.0f}°")

ax = plt.figure(figsize=(6, 6)).add_subplot(projection="polar")
ax.set_theta_zero_location("N")  # 0° en haut
ax.set_theta_direction(-1)       # sens des aiguilles d'une montre, comme le lidar
ax.scatter([math.radians(a) for a in scan.angles], scan.distances, s=3)
ax.set_title("Scan du lidar (distances en m)")
plt.show()

## Caméra

La caméra RealSense donne une image couleur et une carte de profondeur : pour chaque pixel, la distance en mètres. Les pixels sans mesure (trop près, à moins de 30 cm environ, ou surface mal vue) apparaissent en blanc.

In [ ]:
import numpy as np
import pyrealsense2 as rs

UPSIDE_DOWN = True  # sur MobileRobot-1, la caméra est montée tête en bas

pipeline = rs.pipeline()
config = rs.config()
config.enable_stream(rs.stream.color, 640, 480, rs.format.rgb8, 30)
config.enable_stream(rs.stream.depth, 640, 480, rs.format.z16, 30)
profile = pipeline.start(config)
try:
    scale = profile.get_device().first_depth_sensor().get_depth_scale()  # mètres par unité
    align = rs.align(rs.stream.color)  # met la profondeur dans le même cadrage que la couleur
    for _ in range(30):  # laisse l'exposition automatique se régler
        pipeline.wait_for_frames()
    frames = align.process(pipeline.wait_for_frames())
    color = np.asanyarray(frames.get_color_frame().get_data()).copy()
    depth = np.asanyarray(frames.get_depth_frame().get_data()) * scale
finally:
    pipeline.stop()  # libère la caméra
if UPSIDE_DOWN:
    color, depth = color[::-1, ::-1], depth[::-1, ::-1]

print(f"distance au centre de l'image : {depth[240, 320]:.2f} m")
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.5))
left.imshow(color)
left.set_title("Couleur")
image = right.imshow(np.ma.masked_equal(depth, 0), cmap="turbo", vmin=0.2, vmax=4)
right.set_title("Profondeur (m)")
for ax in (left, right):
    ax.axis("off")
fig.colorbar(image, ax=right)
plt.show()

## Et ensuite ?

- Chaque capteur ne peut être utilisé que par **un seul programme à la fois** : fermez-le après usage (`with`, `close()`, `pipeline.stop()`), ou redémarrez le noyau (menu *Kernel*).
- La commande des roues depuis un carnet arrivera avec un **arrêt de sécurité** : les moteurs s'arrêteront d'eux-mêmes si le programme cesse de leur envoyer des consignes.